# motor-video · Video 01 v2 PRO — Imagenes IA + estilo profesional (1080x1920)

1. **Runtime -> Change runtime type -> GPU (T4)** (obligatorio para las imagenes IA).
2. **Runtime -> Ejecutar todo**. Espera ~40-90 min: modelo Wan2.2 (~10GB) + 6 clips IA + montaje.
3. Se descarga `video-01.mp4` solo al final.

**Nivel profesional:** imagenes IA (Wan2.2), voz masculina (Tomas), subtitulos palabra-a-palabra con palabra clave en color, zoom cinematografico con easing cosenoidal (sin temblor), color grading + vineta, cortes cada 2-4s, diseno de loop. Si algo falla en GPU, cada seccion cae a fondo procedural (nunca queda sin video).

In [ ]:
import os, subprocess, sys, json, urllib.request
from pathlib import Path
%pip install -q edge-tts "diffusers>=0.35" transformers accelerate sentencepiece "imageio[ffmpeg]"

WORK = Path('/content/motor-video'); OUT = WORK/'out'; FDIR = WORK/'fonts'
OUT.mkdir(parents=True, exist_ok=True); FDIR.mkdir(exist_ok=True)

# tipografia profesional (Poppins Bold) con fallback DejaVu
FONTFILE = FDIR/'Poppins-Bold.ttf'
try:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/poppins/Poppins-Bold.ttf', FONTFILE)
    print('fuente Poppins Bold OK')
except Exception as e:
    FONTFILE = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf')
    print('Poppins no disponible, fallback DejaVu:', str(e)[:80])

SECCIONES = [
 {'id':'hook',    'dur':5,  'color':'0x0a2a3a','acento':'0x8fe3cf','titulo':'32.4.1',        'sub':'FALLA DE SEGURIDAD'},
 {'id':'problema','dur':9,  'color':'0x2a0a12','acento':'0xff5a5a','titulo':'32.4.1',        'sub':'LA VERSION ACTUAL'},
 {'id':'incognito','dur':9, 'color':'0x141420','acento':'0x9aa4ff','titulo':'MODO INCOGNITO','sub':'NO AISLA SESIONES'},
 {'id':'riesgo',  'dur':9,  'color':'0x2a0a0a','acento':'0xff7a3a','titulo':'RASTREO',       'sub':'SIN TU CONOCIMIENTO'},
 {'id':'solucion','dur':10, 'color':'0x0a2418','acento':'0x4ade80','titulo':'3 AJUSTES',     'sub':'DRAW.IO SEGURO'},
 {'id':'cta',     'dur':8,  'color':'0x0a0a12','acento':'0x8fe3cf','titulo':'PROXIMO VIDEO', 'sub':'EL COMO'},
]
VOZ = 'es-AR-TomasNeural'   # voz masculina rioplatense (encaja con el "podes" del guion)
TEXTO = ('Descubri una falla de seguridad en draw.io. Y lo peor es que afecta a la version 32.4.1, la actual. '
 'En mi auditoria encontre que el modo de incognito no aislA las sesiones como deberia. '
 'Un atacante podria rastrear los diagramas que abres incluso cuando crees que estas protegido. '
 'Pero hay solucion. Con tres ajustes de configuracion podes cerrar esa puerta y dejar tu draw.io realmente seguro. '
 'Te muestro como en el proximo video.')
print('setup OK · objetivo', sum(s['dur'] for s in SECCIONES), 's')

In [ ]:
import asyncio, edge_tts, subprocess
# voz masculina + tiempos de palabra para subtitulos sincronizados
words = []
async def _g():
    c = edge_tts.Communicate(TEXTO, VOZ, boundary='WordBoundary')
    with open(str(OUT/'voz.mp3'),'wb') as f:
        async for ch in c.stream():
            if ch['type']=='audio': f.write(ch['data'])
            elif ch['type']=='WordBoundary':
                words.append({'w':ch['text'],'t0':ch['offset']/1e7,'t1':(ch['offset']+ch['duration'])/1e7})
asyncio.run(_g())
V = float(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(OUT/'voz.mp3')],capture_output=True,text=True).stdout.strip())
peso = sum(s['dur'] for s in SECCIONES)
for s in SECCIONES: s['dur'] = round(s['dur'] * V / peso, 3)
print('voz Tomas:', round(V,2), 's ·', len(words), 'palabras · secciones a', sum(s['dur'] for s in SECCIONES), 's')

# --- subtitulos ASS: palabra a palabra, palabra clave en color de seccion ---
KW = ('32.4.1','FALLA','INCÓGNITO','INCOGNITO','RASTREO','SOLUCION','SOLUCIÓN','AJUSTES','SEGURO','PROTEGIDO','VERSION')
def hex2ass(h):  # '0x8fe3cf' -> '&H00CFE38F&' (BGR de ASS)
    r,g,b = h[4:6], h[2:4], h[0:2]
    return ('&H00%s%s%s&' % (b,g,r)).upper()
def ts(t):
    return '%d:%02d:%05.2f' % (int(t//3600), int(t%3600//60), t%60)
def accent_at(t):
    acc = 0.0
    for s in SECCIONES:
        acc += s['dur']
        if t < acc: return s['acento']
    return SECCIONES[-1]['acento']
ev = []
for i,wd in enumerate(words):
    t0 = wd['t0']
    t1 = min(words[i+1]['t0'] if i+1 < len(words) else V, V)
    if t1 - t0 < 0.20: t1 = min(t0 + 0.20, V)
    up = wd['w'].upper().strip('.,;:¿?¡!…')
    txt = up
    if any(up.startswith(k) for k in KW):
        txt = '{\\1c%s}%s{\\1c&H00FFFFFF&}' % (hex2ass(accent_at(t0)), up)
    ev.append('Dialogue: 0,%s,%s,Cap,,0,0,0,,%s' % (ts(t0), ts(t1), txt))
ass = '[Script Info]\nScriptType: v4.00+\nPlayResX: 1080\nPlayResY: 1920\nWrapStyle: 2\nScaledBorderAndShadow: yes\n' \
 '[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n' \
 'Style: Cap,Poppins,76,&H00FFFFFF,&H00FFFFFF,&H00101010,&H96000000,-1,0,1,5,2,2,90,90,470,1\n' \
 '[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n' + '\n'.join(ev) + '\n'
(OUT/'cap.ass').write_text(ass)
print('cap.ass OK ·', len(ev), 'dialogos')

In [ ]:
# --- B-ROLL IA: Wan2.2-TI2V-5B en la GPU de Colab ---
import torch
GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if GPU else 'NINGUNA — fondos procedurales (pon GPU en Runtime)')

if GPU:
    try:
        try:
            from diffusers import WanVideoPipeline as WP
        except ImportError:
            from diffusers import WanPipeline as WP
        from diffusers.utils import export_to_video

        pipe = WP.from_pretrained('Wan-AI/Wan2.2-TI2V-5B-Diffusers', torch_dtype=torch.float16)
        pipe.enable_model_cpu_offload()
        try: pipe.vae.enable_tiling()
        except Exception: pass

        PROMPTS = {
          'hook':    'extreme close-up of a red glowing warning glitch on dark screen, cybersecurity, teal accents, dramatic volumetric light, cinematic, vertical frame, high detail',
          'problema':'3d metallic padlock cracking apart in slow motion, dark studio, red rim light, sparks, cinematic depth of field, vertical frame, high detail',
          'incognito':'anonymous hooded hacker silhouette facing wall of glowing monitors in dark room, purple blue light, cinematic, vertical frame, high detail',
          'riesgo':  'glowing red surveillance tracking lines converging on a cursor over a dark digital map, data streams, cinematic, vertical frame, high detail',
          'solucion':'emerald green holographic shield with glowing checkmark floating above secure code, dark background, hopeful cinematic light, vertical frame, high detail',
          'cta':     'elegant dark end frame with soft teal particle glow and subtle light streaks, minimal, cinematic, vertical frame, high detail',
        }
        NEG = 'blurry, low quality, watermark, logo, text, letters, distorted, deformed'

        for s in SECCIONES:
            dst = OUT/f"wan_{s['id']}.mp4"
            if dst.exists():
                print('ya existe', dst.name); continue
            try:
                kw = dict(height=832, width=480, num_frames=81, num_inference_steps=30, guidance_scale=5.0)
                gen = torch.manual_seed(42)
                try:
                    frames = pipe(PROMPTS[s['id']], negative_prompt=NEG, generator=gen, **kw).frames[0]
                except TypeError:
                    frames = pipe(PROMPTS[s['id']], generator=gen, **kw).frames[0]
                export_to_video(frames, str(dst), fps=16)
                print('clip IA listo:', dst.name)
            except Exception as e:
                print('fallo clip', s['id'], '→ fallback procedural:', str(e)[:120])
        del pipe; torch.cuda.empty_cache()
    except Exception as e:
        print('No se pudo cargar Wan → fondos procedurales. Error:', str(e)[:200])

In [ ]:
# --- fondos procedurales (fallback garantizado) ---
import os, subprocess
W,H,FPS=1080,1920,30
def esc(s): return s.replace('\\','\\\\').replace(':','\\:').replace('%','\\%')
for sec in SECCIONES:
    if (OUT/f"{sec['id']}.mp4").exists(): continue
    a=sec['acento']; F=str(FONTFILE)
    filt=("drawbox=x=0:y=0:w={W}:h=12:color={a}@0.9:t=fill,"
          "drawbox=x=0:y={Hm}:w={W}:h=12:color={a}@0.9:t=fill,"
          "drawtext=fontfile={F}:text='{t}':fontsize=110:fontcolor=white:borderw=4:bordercolor=black:x=(w-text_w)/2:y=(h-text_h)/2-60,"
          "drawtext=fontfile={F}:text='{s}':fontsize=52:fontcolor={a}:x=(w-text_w)/2:y=(h-text_h)/2+90"
          ).format(W=W,Hm=H-12,a=a,F=F,t=esc(sec['titulo']),s=esc(sec['sub']))
    subprocess.run(['ffmpeg','-y','-f','lavfi','-i',f"color=c={sec['color']}:s={W}x{H}:r={FPS}:d={sec['dur']}",'-vf',filt,'-c:v','libx264','-pix_fmt','yuv420p','-an',str(OUT/f"{sec['id']}.mp4")],check=True)
    print('procedural', sec['id'])

In [ ]:
# --- plano a plano: zoom cinematografico con easing cosenoidal + grading ---
import subprocess
W,H,FPS=1080,1920,30
def shot(src, ds, Z, mode, name):
    N = max(int(round(ds*FPS)), 2)
    ease = '(1-cos(PI*on/%d))/2' % (N-1)
    zexpr = ('1+%s*%s' % (round(Z-1,4), ease)) if mode=='in' else ('%s-%s*%s' % (Z, round(Z-1,4), ease))
    filt = ('fps=30,tpad=stop_mode=clone:stop_duration=8,trim=duration=%s,setpts=PTS-STARTPTS,'
            'scale=2160:3840:force_original_aspect_ratio=increase,crop=2160:3840,'
            "zoompan=z='%s':d=1:x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s=%dx%d:fps=30,"
            'format=yuv420p,eq=contrast=1.06:saturation=1.12:brightness=0.005,vignette=PI/4.5') % (ds, zexpr, W, H)
    subprocess.run(['ffmpeg','-y','-i',str(src),'-filter_complex',filt,'-map','0:v','-c:v','libx264','-crf','19','-preset','medium','-pix_fmt','yuv420p','-an',str(OUT/f's_{name}.mp4')],check=True)
    print('plano', name, mode, ds, 's')

shots = []
for s in SECCIONES:
    ai = OUT/f"wan_{s['id']}.mp4"
    src = ai if (ai.exists() and ai.stat().st_size > 50000) else OUT/f"{s['id']}.mp4"
    d = s['dur']
    if d > 7:   # seccion larga → 2 planos (cambio visual rapido, estilo Shorts)
        dA = round(d*0.55, 3); dB = round(d - dA, 3)
        shots += [(src, dA, 1.16, 'in', s['id']+'_a'), (src, dB, 1.32, 'out', s['id']+'_b')]
    else:
        shots += [(src, d, 1.14, 'in', s['id'])]
for src,d,Z,m,name in shots: shot(src,d,Z,m,name)

lst = OUT/'shots.txt'; lst.write_text(''.join(f"file '{OUT}/s_{n}.mp4'\n" for _,_,_,_,n in shots))
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(lst),'-c','copy',str(OUT/'video.mp4')],check=True)
print('montaje:', len(shots), 'planos →', round(sum(d for _,d,_,_,_ in shots),2), 's')

In [ ]:
# --- subtitulos quemados + voz + faststart ---
import subprocess
video = OUT/'video.mp4'
cap = OUT/'video_cap.mp4'
vf = 'ass=%s:fontsdir=%s' % (OUT/'cap.ass', FDIR)
r = subprocess.run(['ffmpeg','-y','-i',str(video),'-vf',vf,'-c:v','libx264','-crf','19','-preset','medium','-pix_fmt','yuv420p','-an',str(cap)],capture_output=True,text=True)
if r.returncode != 0:
    print('ass no disponible, video sin subtitulos:', r.stderr[-150:]); cap = video
subprocess.run(['ffmpeg','-y','-i',str(cap),'-i',str(OUT/'voz.mp3'),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-af','apad','-t',str(V),'-movflags','+faststart',str(OUT/'video-01.mp4')],check=True)
print('video-01.mp4 listo ·', round(V,2), 's · voz + subtitulos + faststart')

In [ ]:
import subprocess, json
p=str(OUT/'video-01.mp4')
info=json.loads(subprocess.run(['ffprobe','-v','error','-show_entries','stream=width,height,codec_type:format=duration','-of','json',p],capture_output=True,text=True).stdout)
dur=float(info['format']['duration']); st={s['codec_type']:s for s in info['streams']}
w,h=st['video']['width'],st['video']['height']; audio='audio' in st
faststart=b'moov' in open(p,'rb').read(262144)
sync=abs(dur - V) <= 0.5
ok=(w,h)==(1080,1920) and dur<=60 and audio and faststart and sync
print(json.dumps({'res':f'{w}x{h}','dur':round(dur,1),'voz':round(V,1),'audio':audio,'faststart':faststart,'sync':sync,'QA':'PASS' if ok else 'FAIL'}))
assert ok, 'QA FAIL'

In [ ]:
from google.colab import files
files.download(str(OUT/'video-01.mp4'))